# Companion Lab 02 — Train a Real Chest X-Ray CNN on PneumoniaMNIST

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/5x5x5x5/taihls/blob/curriculum-v2/notebooks/02-medical-images-cnn.ipynb)

In **Chapter 3.5**, we wrote a 2D convolution filter from scratch in NumPy. Here we train a real **Convolutional Neural Network (CNN)** in PyTorch on **PneumoniaMNIST** — thousands of real pediatric chest radiographs labeled `0 = normal` vs. `1 = pneumonia` — and grade it using real clinical metrics (**sensitivity, specificity, AUROC, and error inspection**).

> **Enable GPU on Colab:** *Runtime → Change runtime type → T4 GPU*.

In [ ]:
!pip install -q medmnist torch torchvision scikit-learn matplotlib

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import torchvision.transforms as T
import matplotlib.pyplot as plt
import medmnist
from medmnist import INFO
from sklearn.metrics import confusion_matrix, roc_auc_score

torch.manual_seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

info = INFO['pneumoniamnist']
DataClass = getattr(medmnist, info['python_class'])
transform = T.Compose([T.ToTensor()])

train_ds = DataClass(split='train', transform=transform, download=True)
val_ds   = DataClass(split='val',   transform=transform, download=True)
test_ds  = DataClass(split='test',  transform=transform, download=True)
print(f'Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)} scans (28x28 grayscale)')

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=256, shuffle=False)

## 1. Always look at the scans first (and check class balance!)

In [ ]:
y_tr_all = train_ds.labels.ravel()
y_te_all = test_ds.labels.ravel()
print(f'Train pneumonia prevalence: {y_tr_all.mean():.1%} | Test prevalence: {y_te_all.mean():.1%}')

fig, axes = plt.subplots(1, 6, figsize=(11, 2.2))
for ax, (img, lbl) in zip(axes, train_ds):
    ax.imshow(img.squeeze(), cmap='gray')
    ax.set_title('Pneumonia' if int(lbl) == 1 else 'Normal', fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()

## 2. Build and train a compact 2-block CNN

Notice the two `nn.Conv2d` layers learning 16 and 32 small `3×3` convolutional kernels automatically via backpropagation — the exact scaled-up cousin of our hand-coded `conv2d()` in Chapter 3.5!

In [ ]:
model = nn.Sequential(
    nn.Conv2d(1, 16, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),  # 28->14
    nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2), # 14->7
    nn.Flatten(),
    nn.Linear(32 * 7 * 7, 64), nn.ReLU(),
    nn.Linear(64, 2),
).to(device)

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(6):
    model.train()
    total_loss = 0.0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.squeeze().long().to(device)
        optimizer.zero_grad()
        loss = loss_fn(model(imgs), labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f'Epoch {epoch+1}/6 | avg train loss = {total_loss / len(train_loader):.3f}')

## 3. Clinical audit: Confusion matrix, Sensitivity, Specificity, AUROC, and Error Inspection

In [ ]:
model.eval()
all_probs, all_labels, all_imgs = [], [], []
with torch.no_grad():
    for imgs, labels in test_loader:
        logits = model(imgs.to(device))
        probs = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
        all_probs.append(probs)
        all_labels.append(labels.squeeze().numpy())
        all_imgs.append(imgs.squeeze(1).numpy())

probs = np.concatenate(all_probs)
y_true = np.concatenate(all_labels)
X_imgs = np.concatenate(all_imgs)
preds = (probs >= 0.50).astype(int)

tn, fp, fn, tp = confusion_matrix(y_true, preds, labels=[0, 1]).ravel()
print(f'Test Accuracy   : {(preds == y_true).mean():.1%}')
print(f'Test AUROC      : {roc_auc_score(y_true, probs):.3f}')
print(f'Sensitivity     : {tp / (tp + fn):.1%}  (missed {fn} pneumonia scans!)')
print(f'Specificity     : {tn / (tn + fp):.1%}  ({fp} false alarms on normal scans)')

# Visualize worst mistakes (False Negatives and False Positives)
wrong = np.where(preds != y_true)[0][:6]
fig, axes = plt.subplots(1, len(wrong), figsize=(12, 2.3))
for ax, idx in zip(axes, wrong):
    ax.imshow(X_imgs[idx], cmap='gray')
    kind = 'FN (missed!)' if y_true[idx] == 1 else 'FP (false alarm)'
    ax.set_title(f'{kind}\nP={probs[idx]:.2f}', fontsize=8.5, color='#D55E00')
    ax.axis('off')
plt.suptitle('Inspecting actual CNN errors on held-out pediatric chest X-rays', y=1.05)
plt.tight_layout()
plt.show()

## 4. Modify & Make (PRIMM challenges)

1. **Tune the clinical threshold (Chapter 2.4):** Change `preds = (probs >= 0.50)` to `0.25` and `0.75`. Watch how sensitivity and specificity trade off without retraining!
2. **Inspect learned first-layer kernels:** Plot the 16 learned `3×3` filters from `model[0].weight.detach().cpu().numpy()`. Do any resemble the edge/spot filters from Chapter 3.5?
3. **Shortcut reflection (Chapter 3.5):** Remember that `train_ds` is ~74% pneumonia — much higher than a general clinic! What happens to PPV if deployed in a clinic where pneumonia prevalence is only 5%?